In [48]:
import os
import random

import json

import pandas as pd

from dotenv import load_dotenv
import time
from datetime import datetime
from openai import OpenAI

from modules.corpus import get_gt_dict
from config.data_config import DATA_CONFIG, FEW_SHOT_PATH, BIORED_GUIDELINES_PATH
from modules.parsing import outputs_to_eval_tuples
from modules.evaluation import evaluate_extractions

## Configuration

### Development Configuration

In [49]:
# ---------- Set seed ----------
load_dotenv()

# ---------- Set seed to 42 ----------
SEED = int(os.getenv("SEED", 42))
random.seed(SEED)

# ---------- Print confirmation ----------
print(
    f"Set random seed to {SEED} at "
    f"{datetime.now().strftime('%Y-%m-%d %H:%M')}"
)

Set random seed to 42 at 2026-08-30 17:44


### Pipeline Configuration

In [63]:
# ---------- Manual EPI config. entry ----------
DATASET_SPLIT = "biored_test"

# ---------- Config. whether EPI config's API should be called again ----------
CALL_API = False

## Corpus Loading

### Load Abstracts

In [64]:
# ---------- Get abstracts path ----------
ABSTRACTS_FULL_PATH = DATA_CONFIG[DATASET_SPLIT]["abstracts"]["abstracts_full"]
ABSTRACTS_EVAL_PATH = DATA_CONFIG[DATASET_SPLIT]["abstracts"]["abstracts_eval"]

# ---------- Load corpus from path ----------
abstracts_full = pd.read_csv(ABSTRACTS_FULL_PATH)
abstracts_eval = pd.read_csv(ABSTRACTS_EVAL_PATH)

# ---------- Set output dir. ----------
OUTPUT_DIRECTORY = DATA_CONFIG[DATASET_SPLIT]["output_directory"]

# ---------- Set PMID as index ----------
abstracts_full.set_index("pmid", inplace=True)
abstracts_eval.set_index("pmid", inplace=True)

print(f"Abstracts full length: {len(abstracts_full)}")
print(f"Abstracts eval. length: {len(abstracts_eval)}")

Abstracts full length: 100
Abstracts eval. length: 100


### Import Few-Shot Examples

In [65]:
with open(FEW_SHOT_PATH, "r") as file:
    few_shot_block = file.read()

print("Successfully imported few shot block.")
print(f"{few_shot_block[:500]}")

Successfully imported few shot block.
## EXAMPLE 1:

### Abstract:

Massive urinary protein excretion has been observed after conversion from calcineurin inhibitors to mammalian target of rapamycin (mToR) inhibitors, especially sirolimus, in renal transplant recipients with chronic allograft nephropathy. Because proteinuria is a major predictive factor of poor transplantation outcome, many studies focused on this adverse event during the past years. Whether proteinuria was due to sirolimus or only a consequence of calcineurin inhibi


### Import BioRED Extraction Guidelines

In [66]:
with open(BIORED_GUIDELINES_PATH, "r") as file:
    biored_guidelines = file.read()

print("Successfully imported official BioRED guidelines.")
print(f"{biored_guidelines[:500]}")

Successfully imported official BioRED guidelines.
## Guideline of the entities

### General rules
- Annotate all the spans of all the six concept types.
- The full text can be accessed to clarify the concept spans and identifiers.
- The abbreviation and its long form should be annotated separately if possible. prostaglandin E2 (PGE2) in the text, “prostaglandin E2” and “PGE2” should be both annotated to chemicals with the same identifier (D015232).
- Annotate both the full name and abbreviation in one entity, if the boundary of the entity cover


# LLM-API Extraction

### Extraction Configuration

In [67]:
with open("config/prompt.txt", "r") as file:
    prompt_template = file.read()

print("Successfully imported prompt.")
print(f"{prompt_template[:500]}")

Successfully imported prompt.
# TASK\n\nYou are an expert biomedical information extraction system.\n\nYour task is to extract biomedical entities and relationships from the abstract below exactly as they would be annotated under the official BioRED annotation guidelines.\n\nThe official BioRED annotation guidelines are provided below.\n\n# BioRED Annotation Guidelines\n\n{biored_ext_guidelines}\n\n# Allowed Entity Types\n\nOutput ONLY the following entity types:\n\n- ChemicalEntity\n- DiseaseOrPhenotypicFeature\n- GeneOrGen


### Initiate Client

In [68]:
client = OpenAI(
    api_key=os.getenv("OPEN_AI_KEY"),
    timeout=120,
    max_retries=2
)

In [69]:
# ---------- API call ----------
if CALL_API:
    print(f"Running API call for 'epi_009' on '{DATASET_SPLIT}'...")

    start_time = time.perf_counter()

    outputs = []

    running_tokens = 0

    current_abstract_num = 1
    total_abstracts_num = abstracts_full.shape[0]

    for index, row in abstracts_full.iterrows():

        abstract = row["abstract"]
        title = row["title"]

        prompt = (
            prompt_template
            .replace("{title}", title)
            .replace("{abstract}", abstract)
            .replace("{few_shot_block}", few_shot_block)
            .replace("{biored_ext_guidelines}", biored_guidelines)
        )

        response = client.responses.create(
            model="gpt-5.6-luna",
            input=prompt
        )

        outputs.append({
            "pmid": index,
            "output": response.output_text,
            "prompt_tokens": response.usage.input_tokens,
            "completion_tokens": response.usage.output_tokens,
            "total_tokens": response.usage.total_tokens
        })

        running_tokens += response.usage.total_tokens

        progress_time = time.perf_counter() - start_time
        avg_time_per_abstract = progress_time / current_abstract_num
        est_time_remaining = avg_time_per_abstract * (total_abstracts_num - current_abstract_num)
        minutes = int(est_time_remaining // 60)
        seconds = est_time_remaining % 60

        print(
            f"\rCompleteled extraction for abstract {current_abstract_num} of {total_abstracts_num} "
            f"({current_abstract_num / total_abstracts_num * 100:.1f}%; "
            f"Est. time remaining: {minutes}m {seconds:.0f}s).",
            end="",
            flush=True
        )

        current_abstract_num += 1

    extraction_info = {
        "dataset": DATASET_SPLIT,
        "model": "gpt_5.6_luna",
        "prompt_template": prompt_template,
        "time_complete": datetime.now().isoformat(),
        "time_elapsed": time.perf_counter() - start_time,
        "outputs": outputs
    }

    # ---------- Export extraction_info dict. as JSON ----------
    with open(f"{OUTPUT_DIRECTORY}{DATASET_SPLIT}_extraction_log.json", "w") as f:
        json.dump(extraction_info, f, indent=2)

    print(f"Successfully ran API call at {datetime.now().strftime('%Y-%m-%d %H:%M')}")
else:

    existing_log =  f"{OUTPUT_DIRECTORY}{DATASET_SPLIT}_extraction_log.json"

    try:
        with open(existing_log, "r") as f:
            extraction_info = json.load(f)
    except:
        raise ValueError(
            f"Extraction log cannot be imported - "
            f"'{existing_log}' log does not exist."
        )

### Parse `output` to entity and relation dictionaries
* Includes `parse_failures`

In [70]:
predictions_entities, predictions_relations, parse_failures = outputs_to_eval_tuples(extraction_info["outputs"])

extraction_dict = {
    "ner": predictions_entities,
    "re": predictions_relations
}

Parsed 100 extractions, 0 failed to parse as JSON


### Export entities and relations as JSON (list of dicts)

In [71]:
entities_export = [
    {"pmid": pmid, "text": text, "type": entity_type}
    for pmid, text, entity_type in predictions_entities
]

relations_export = [
    {"pmid": pmid, "source": source, "relation": relation, "target": target}
    for pmid, source, relation, target in predictions_relations
]

with open(f"{OUTPUT_DIRECTORY}{DATASET_SPLIT}_predictions.json", "w") as f:
    json.dump({"entities": entities_export, "relations": relations_export}, f, indent=2, default=str)

# Extraction Evaluation

### Load Ground Truths

In [72]:
# ---------- Manually Set Ground Truth Paths ----------
NER_GT_PATH = DATA_CONFIG[DATASET_SPLIT]["ground_truths"]["ner"]
RE_GT_PATH = DATA_CONFIG[DATASET_SPLIT]["ground_truths"]["re"]

# ---------- Fetch BioRED Ground Truths ----------
ner_gt = pd.read_csv(NER_GT_PATH)
re_gt = pd.read_csv(RE_GT_PATH)

### Parse GT CSVs Into GT Dictionaries

In [73]:
gt_dict = get_gt_dict(ner_gt, re_gt, abstracts_eval)

print("Successfully parsed ground truth CSVs to dict[dict] format")

Successfully parsed ground truth CSVs to dict[dict] format


In [74]:
# ---------- Evaluation results ----------
eval_results = evaluate_extractions(predictions_entities, predictions_relations, gt_dict)

print("Entity metrics:")
print(f"  Precision: {eval_results['entity']['point_estimate']['precision']:.4f} "
      f"(95% CI: {eval_results['entity']['ci']['precision']})")
print(f"  Recall:    {eval_results['entity']['point_estimate']['recall']:.4f} "
      f"(95% CI: {eval_results['entity']['ci']['recall']})")
print(f"  F1:        {eval_results['entity']['point_estimate']['f1']:.4f} "
      f"(95% CI: {eval_results['entity']['ci']['f1']})")

print("\nRelation metrics:")
print(f"  Precision: {eval_results['relation']['point_estimate']['precision']:.4f} "
      f"(95% CI: {eval_results['relation']['ci']['precision']})")
print(f"  Recall:    {eval_results['relation']['point_estimate']['recall']:.4f} "
      f"(95% CI: {eval_results['relation']['ci']['recall']})")
print(f"  F1:        {eval_results['relation']['point_estimate']['f1']:.4f} "
      f"(95% CI: {eval_results['relation']['ci']['f1']})")

Entity metrics:
  Precision: 0.9224 (95% CI: (0.9020407829274679, 0.9414360415931149))
  Recall:    0.7558 (95% CI: (0.727151300837496, 0.7827118117539991))
  F1:        0.8308 (95% CI: (0.8100130272027876, 0.8512514533513496))

Relation metrics:
  Precision: 0.5181 (95% CI: (0.4596774193548387, 0.5779079178940829))
  Recall:    0.5005 (95% CI: (0.44515300868837154, 0.5562685336281167))
  F1:        0.5092 (95% CI: (0.4605801121630585, 0.5557619342721817))


### Export Extraction Performance Results

In [75]:
with open(f"{OUTPUT_DIRECTORY}{DATASET_SPLIT}_results.json", "w") as file:
    json.dump(eval_results, file, indent=2)

print(f"Exported '{DATASET_SPLIT}_results.json' to '{OUTPUT_DIRECTORY}'")

Exported 'biored_test_results.json' to '../data/results/final/biored_test/'
